In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
sys.path.append('../helpers')
import functions as vh

In [2]:
states_df = pd.read_parquet('../data/frenet_states.parquet')
track_df = pd.read_csv('../data/track_centerline.csv')

velocity_lookup = pd.read_csv('../data/velocity_lookup.csv')
v_target_dict = dict(zip(velocity_lookup['s'], velocity_lookup['v']))

states_df = states_df.sort_values('s').reset_index(drop=True)

all_s = track_df['s'].values
layers = sorted(states_df['s'].unique())
edges_list = []

for i in tqdm(range(len(layers) - 1), desc="Generating Layer Edges"):

    layer1 = states_df[states_df['s'] == layers[i]].reset_index(drop=True)
    layer2 = states_df[states_df['s'] == layers[i + 1]].reset_index(drop=True)

    x1 = layer1['E'].values[:, None]
    x2 = layer2['E'].values[None, :]
    y1 = layer1['N'].values[:, None]
    y2 = layer2['N'].values[None, :]

    arc_length = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    v1 = layer1['v'].values[:, None]
    v2 = layer2['v'].values[None, :]

    v1_b = np.broadcast_to(v1, (len(layer1), len(layer2)))
    v2_b = np.broadcast_to(v2, (len(layer1), len(layer2)))

    arc_length_safe = np.maximum(arc_length, 1e-6)
    acceleration = (v2**2 - v1**2) / (2 * arc_length_safe)
    heading = np.degrees(np.arctan2(y2 - y1, x2 - x1))

    from_ids = layer1['state_id'].values[:, None].repeat(len(layer2), axis=1)
    to_ids = layer2['state_id'].values[None, :].repeat(len(layer1), axis=0)

    mask = ~((v1 == 0) & (v2 == 0))

    from_ids = from_ids.flatten()[mask.flatten()]
    to_ids = to_ids.flatten()[mask.flatten()]
    acceleration = acceleration.flatten()[mask.flatten()]
    arc_length_masked = arc_length.flatten()[mask.flatten()]
    heading = heading.flatten()[mask.flatten()]
    avg_velocity = ((v1 + v2) / 2).flatten()[mask.flatten()]
    v1_flat = v1_b.flatten()[mask.flatten()]
    v2_flat = v2_b.flatten()[mask.flatten()]

    expected_edges = len(layer1) * len(layer2)
    actual_edges = np.sum(mask)

    if actual_edges != expected_edges:
        print(f"Warning: Layer {i}->{i+1} missing {expected_edges - actual_edges} masked edges.")

    to_s_values = layer2.set_index('state_id').loc[to_ids]['s'].values
    v_target = np.array([v_target_dict.get(s_val, 0) for s_val in to_s_values])

    cost, m_H2_edge = vh.compute_edge_costs(
        acceleration,
        v1_flat,
        v2_flat,
        arc_length_masked,
        v_target
    )

    time_per_edge = arc_length_masked / np.maximum(avg_velocity, 1e-6)

    edges_list.append(pd.DataFrame({
        'from_state_id': from_ids,
        'to_state_id': to_ids,
        'from_s': layer1.set_index('state_id').loc[from_ids]['s'].values,
        'to_s': layer2.set_index('state_id').loc[to_ids]['s'].values,
        'from_E': layer1.set_index('state_id').loc[from_ids]['E'].values,
        'to_E': layer2.set_index('state_id').loc[to_ids]['E'].values,
        'from_N': layer1.set_index('state_id').loc[from_ids]['N'].values,
        'to_N': layer2.set_index('state_id').loc[to_ids]['N'].values,
        'from_v': layer1.set_index('state_id').loc[from_ids]['v'].values,
        'to_v': layer2.set_index('state_id').loc[to_ids]['v'].values,
        'from_d': layer1.set_index('state_id').loc[from_ids]['d'].values,
        'to_d': layer2.set_index('state_id').loc[to_ids]['d'].values,
        'arc_length_path': arc_length_masked,
        'acceleration_long': acceleration,
        'heading': heading,
        'time_sec': time_per_edge,
        'cost': cost,
        'm_H2': m_H2_edge
    }))

edges_df = pd.concat(edges_list, ignore_index=True)
edges_df.to_parquet('../data/frenet_edges.parquet', index=False)

Generating Layer Edges: 100%|██████████| 187/187 [00:00<00:00, 389.59it/s]
